# Convert ReacDiff2D Data to Symmetry Discovery Format

This notebook converts the 2D reaction-diffusion data from `/data/symmetry-sr/ReacDiff2D/rd_noise0.0005.h5` to the HDF5 format required by the symmetry discovery tool.

In [1]:
import h5py
import numpy as np
import json

In [2]:
# Examine source file structure
src_path = "/data/symmetry-sr/ReacDiff2D/rd_noise0.0005.h5"

with h5py.File(src_path, 'r') as f:
    print("Keys:", list(f.keys()))
    print()
    for key in f.keys():
        data = f[key]
        if hasattr(data, 'shape'):
            print(f"{key}: shape={data.shape}, dtype={data.dtype}")

Keys: ['E_t', 'E_x', 'E_xx', 'E_xy', 'E_y', 'E_yy', 'I_t', 'I_x', 'I_xx', 'I_xy', 'I_y', 'I_yy', 'R', 'spatial_grid', 't', 'u', 'u_t', 'u_x', 'u_xx', 'u_xy', 'u_y', 'u_yy']

E_t: shape=(128, 128, 200), dtype=float64
E_x: shape=(128, 128, 200), dtype=float64
E_xx: shape=(128, 128, 200), dtype=float64
E_xy: shape=(128, 128, 200), dtype=float64
E_y: shape=(128, 128, 200), dtype=float64
E_yy: shape=(128, 128, 200), dtype=float64
I_t: shape=(128, 128, 200), dtype=float64
I_x: shape=(128, 128, 200), dtype=float64
I_xx: shape=(128, 128, 200), dtype=float64
I_xy: shape=(128, 128, 200), dtype=float64
I_y: shape=(128, 128, 200), dtype=float64
I_yy: shape=(128, 128, 200), dtype=float64
R: shape=(128, 128, 200), dtype=float64
spatial_grid: shape=(128, 128, 2), dtype=float64
t: shape=(200,), dtype=float64
u: shape=(128, 128, 200, 2), dtype=float64
u_t: shape=(128, 128, 200, 2), dtype=float64
u_x: shape=(128, 128, 200, 2), dtype=float64
u_xx: shape=(128, 128, 200, 2), dtype=float64
u_xy: shape=(128,

In [3]:
def convert_reacdiff2d_to_h5(
    src_path: str,
    dst_path: str,
    dependent_var_names: list = None,
    feature_vars: list = None,
    target_vars: list = None,
):
    """
    Convert ReacDiff2D data to symmetry discovery format.
    
    Source data structure:
        - u: shape (128, 128, 200, 2) - [u, v] stacked in last dim
        - u_t, u_x, u_y, u_xx, u_yy, u_xy: shape (128, 128, 200, 2) - derivatives stacked
        - spatial_grid: shape (128, 128, 2) - [x, y] coordinates
        - t: shape (200,) - time values
    
    Target format:
        - Each variable as separate dataset with shape (N_samples,)
        - Flattened from (128*128*200,) = 3,276,800 samples
        - Attributes: independent_variables, dependent_variables, feature_variables, target_variables
    """
    
    if dependent_var_names is None:
        dependent_var_names = ["u", "v"]
    
    with h5py.File(src_path, 'r') as src:
        # Get dimensions from u array: (nx, ny, nt, 2)
        nx, ny, nt, n_vars = src['u'].shape
        n_samples = nx * ny * nt
        print(f"Source data: {nx}x{ny} spatial, {nt} timesteps, {n_vars} variables = {n_samples} samples")
        
        # Load and flatten dependent variables (u, v from u[:,:,:,0] and u[:,:,:,1])
        u_data = src['u'][:, :, :, 0].flatten()  # (nx, ny, nt) -> (n_samples,)
        v_data = src['u'][:, :, :, 1].flatten()
        
        # Load and flatten derivatives (all from u_* arrays)
        u_t = src['u_t'][:, :, :, 0].flatten()
        v_t = src['u_t'][:, :, :, 1].flatten()
        u_x = src['u_x'][:, :, :, 0].flatten()
        v_x = src['u_x'][:, :, :, 1].flatten()
        u_y = src['u_y'][:, :, :, 0].flatten()
        v_y = src['u_y'][:, :, :, 1].flatten()
        u_xx = src['u_xx'][:, :, :, 0].flatten()
        v_xx = src['u_xx'][:, :, :, 1].flatten()
        u_yy = src['u_yy'][:, :, :, 0].flatten()
        v_yy = src['u_yy'][:, :, :, 1].flatten()
        u_xy = src['u_xy'][:, :, :, 0].flatten()
        v_xy = src['u_xy'][:, :, :, 1].flatten()
        
        print(f"Flattened u shape: {u_data.shape}")
        print(f"Flattened u_t shape: {u_t.shape}")
    
    # Default feature and target variables for reaction-diffusion PDE
    # For dx/dt = f(x, x_xx, x_yy, ...), features include state and spatial derivatives
    if feature_vars is None:
        feature_vars = ["u", "v", "u_xx", "u_yy", "v_xx", "v_yy"]
    
    if target_vars is None:
        target_vars = ["u_t", "v_t"]
    
    # Build data dictionary
    data_dict = {
        "u": u_data, "v": v_data,
        "u_t": u_t, "v_t": v_t,
        "u_x": u_x, "v_x": v_x,
        "u_y": u_y, "v_y": v_y,
        "u_xx": u_xx, "v_xx": v_xx,
        "u_yy": u_yy, "v_yy": v_yy,
        "u_xy": u_xy, "v_xy": v_xy,
    }
    
    # Write to destination
    with h5py.File(dst_path, 'w') as dst:
        # Write all variables that appear in features or targets
        all_vars = set(feature_vars + target_vars)
        for var_name in all_vars:
            if var_name in data_dict:
                dst.create_dataset(var_name, data=data_dict[var_name].astype(np.float32))
                print(f"Created dataset '{var_name}': shape {data_dict[var_name].shape}")
            else:
                raise ValueError(f"Variable '{var_name}' not found in data")
        
        # Store metadata as JSON-encoded attributes
        dst.attrs['independent_variables'] = json.dumps(['t', 'x', 'y'])
        dst.attrs['dependent_variables'] = json.dumps(dependent_var_names)
        dst.attrs['feature_variables'] = json.dumps(feature_vars)
        dst.attrs['target_variables'] = json.dumps(target_vars)
        
        print(f"\nSaved to: {dst_path}")
        print(f"  independent_variables: ['t', 'x', 'y']")
        print(f"  dependent_variables: {dependent_var_names}")
        print(f"  feature_variables: {feature_vars}")
        print(f"  target_variables: {target_vars}")

In [4]:
# Convert with default settings
# Features: E, I, E_xx, E_yy, I_xx, I_yy (state + Laplacian for diffusion)
# Targets: E_t, I_t (time derivatives)

convert_reacdiff2d_to_h5(
    src_path="/data/symmetry-sr/ReacDiff2D/rd_noise0.0005.h5",
    dst_path="/home/ubuntu/LLM-SR/agentsr/src/tools/symmetry_discovery/rd.h5",
)

Source data: 128x128 spatial, 200 timesteps, 2 variables = 3276800 samples
Flattened u shape: (3276800,)
Flattened u_t shape: (3276800,)
Created dataset 'u_xx': shape (3276800,)
Created dataset 'v_yy': shape (3276800,)
Created dataset 'v_t': shape (3276800,)
Created dataset 'u': shape (3276800,)
Created dataset 'u_yy': shape (3276800,)
Created dataset 'u_t': shape (3276800,)
Created dataset 'v_xx': shape (3276800,)
Created dataset 'v': shape (3276800,)

Saved to: /home/ubuntu/LLM-SR/agentsr/src/tools/symmetry_discovery/rd.h5
  independent_variables: ['t', 'x', 'y']
  dependent_variables: ['u', 'v']
  feature_variables: ['u', 'v', 'u_xx', 'u_yy', 'v_xx', 'v_yy']
  target_variables: ['u_t', 'v_t']


In [ ]:
# Verify the converted file
with h5py.File("/data/symmetry-sr/ReacDiff2D/rd_noise0.0005_converted.h5", 'r') as f:
    print("Datasets:", list(f.keys()))
    print("\nShapes:")
    for key in f.keys():
        print(f"  {key}: {f[key].shape}")
    print("\nAttributes:")
    for k, v in f.attrs.items():
        print(f"  {k}: {v}")

In [ ]:
# Test loading with the symmetry discovery tool's ODEDataset
import sys
sys.path.insert(0, '.')
from tool import ODEDataset

dataset = ODEDataset("/data/symmetry-sr/ReacDiff2D/rd_noise0.0005_converted.h5")
print(f"\nDataset loaded successfully!")
print(f"  Number of samples: {len(dataset)}")
print(f"  Feature dim: {dataset.get_feature_dim()}")
print(f"  Target dim: {dataset.get_target_dim()}")
print(f"  Num dependent vars: {dataset.get_num_dependent_vars()}")
print(f"  Derivative names: {dataset.get_derivative_names()}")

# Check a sample
sample = dataset[0]
print(f"\nSample structure:")
for key, value in sample.items():
    print(f"  {key}: shape {value.shape}")